# Simulation Calibration: Can the Digital Twin Correct Itself?

*Use case 2 of 3, about 20 minutes.*

OpenDT predicts the power draw of each host with a power model.
Like many state-of-the-art simulators in the field, OpenDC uses a deterministic and static simulation strategy.
However, static simulation models can drift and introduce errors as the time horizon increases: hardware behavior
varies with temperature, aging, and firmware updates, while workload characteristics evolve over time.
To mitigate this problem, OpenDT adds a Self-Calibrator, which measures the difference between simulation-predicted
results and actual telemetry, then continuously adjusts the simulation model.

In this notebook, we (1) look at the power model and its calibration parameter r, (2) calibrate r with a grid search,
and (3) replay the self-calibrating digital twin over one week, hour by hour.

Run the cells in order (Shift+Enter). When short on time, skip the parts marked *optional*.

In [ ]:
import numpy as np
import pandas as pd
from opendt_tutorial import *

check_setup()

## 1. The Power Model

OpenDT models the power draw of a host with the OpenDC analytical formula
$P(u) = P_{idle} + (P_{max} - P_{idle})(2u - u^r)$.
Here, $u$ is CPU utilization, $P_{idle}$ and $P_{max}$ represent idle and max power, and $r$ is the calibration
parameter.
The SURF topology in OpenDT uses $P_{idle}$ = 25 W, $P_{max}$ = 174 W, and $r$ = 4.

In [ ]:
u = np.linspace(0, 1, 101)  # CPU utilization, from 0 (idle) to 1 (full load)
P_IDLE, P_MAX = 25, 174  # watts

curves = {f"r = {r}": P_IDLE + (P_MAX - P_IDLE) * (2 * u - u**r) for r in [2, 4, 6, 9]}
ax = pd.DataFrame(curves, index=u * 100).plot(figsize=(10, 4))
ax.axhline(P_MAX, color="black", linewidth=0.8, linestyle="--", label="$P_{max}$")
ax.set(xlabel="CPU Utilization [%]", ylabel="Power Draw [W]", ylim=(0, None))
legend_above(ax)

*Question:* in notebook 1, OpenDT predicts less power than the ground truth (under-estimation). To reduce the
error, should we increase or decrease r?

<details><summary>Answer</summary>

Increase r. At the same CPU utilization, a higher r gives a higher power draw (e.g., at 50% utilization,
165 W with r = 4 vs. 174 W with r = 9).
</details>

*Question (optional):* for r > 2, the curves go above $P_{max}$. Can a host draw more than its maximum power?

<details><summary>Answer</summary>

No. The formula is an empirical model, not a physical law; with r > 2, it exceeds $P_{max}$ below full load
(e.g., 202 W at 80% utilization with r = 4).
</details>

## 2. Calibrating r with a Grid Search

The Self-Calibrator employs a grid search strategy over the power model's parameter space: it evaluates a set of
candidate values by running simulations and comparing the results against the ground truth.
The value with the lowest MAPE wins.
We simulate the full week (6-13 October 2022, 7,850 jobs) once for each r from 3 to 9, the range of experiment E2 in
the OpenDT paper (`config/experiments/experiment_2.yaml`). This takes up to a minute.

In [ ]:
R_VALUES = np.linspace(3, 9, 7)  # r = 3, 4, ..., 9
sweep = simulate_many("r", R_VALUES, until=TRACE_END)  # one simulation of the full week per r


def error(run):
    """Error of a simulation at each minute [%]: |ground truth - prediction| / ground truth."""
    both = align(run)  # ground truth and prediction on the same 1-minute grid
    return (both["measured"] - both["simulated"]).abs() / both["measured"] * 100


errors = pd.DataFrame({r: error(run) for r, run in sweep.items()})  # one column per r
errors.mean().rename_axis("r").rename("MAPE [%]").round(1).to_frame()

The MAPE is the mean of these errors over all minutes, as in the Self-Calibrator.

*Question:* which r fits the week best? Is it r = 4, the value in the SURF topology?

<details><summary>Answer</summary>

r = 7, with a MAPE of 4.5% vs. 6.4% for r = 4.
</details>

## 3. Drift

One r for the whole week assumes that the datacenter and its workload do not change. We now identify the best r for
each day of the week.

In [ ]:
best_r_per_day = errors.resample("24h", origin=TRACE_START).mean().idxmin(axis=1)
best_r_per_day

*Question:* is the best r the same every day? What does this mean for a digital twin?

<details><summary>Answer</summary>

No, the best r changes between 5 and 9 from day to day.
Static simulation models drift; therefore, OpenDT recalibrates continuously, during operation.
</details>

## 4. The Self-Calibrating Digital Twin

In experiment E2, OpenDT predicts the datacenter one hour at a time (windows of operation of 60 minutes).
After each hour, the Self-Calibrator picks the r with the lowest MAPE over the last hour, and OpenDT predicts the next
hour with this r (in the live twin, a grid search takes time, so the new r may arrive later).
Because OpenDC is deterministic, we replay this process from the simulations above, without new simulations.

*Question:* the self-calibrating digital twin only knows the last hour. Can it be more accurate than r = 7, the best r
of the whole week? Write down a guess, then run the next cell.

In [ ]:
hourly = errors.resample("1h").mean()  # MAPE of each r, per hour


def self_calibrate(hourly, picked):
    """MAPE per hour when each hour uses the r picked after the previous hour (r = 4 first)."""
    used = picked.shift(1, fill_value=4.0)
    return pd.Series([hourly.loc[hour, r] for hour, r in used.items()], index=hourly.index)


best_r = hourly.idxmin(axis=1)  # the r with the lowest MAPE in each hour
no_calibration = hourly[4.0]
with_calibration = self_calibrate(hourly, best_r)

for name, mape_per_hour in [("No Calibration", no_calibration), ("With Calibration", with_calibration)]:
    print(f"{name}: MAPE {mape_per_hour.mean():.1f}%, hours within 10%: {(mape_per_hour < 10).mean():.0%}")

In [ ]:
both = pd.DataFrame({"No Calibration": no_calibration, "With Calibration": with_calibration})
ax = both.plot(figsize=(10, 4), color=["#E69F00", "#009E73"])
ax.axhline(10, color="red", linewidth=0.8, label="NFR Threshold (10%)")
ax.axhline(7.86, color="#338ec1", linewidth=0.8, label="FootPrinter (7.86%)")
ax.set(xlabel="Time [UTC]", ylabel="MAPE [%]", ylim=(0, None))
legend_above(ax)

*Question:* how do these numbers compare with the OpenDT paper, where calibration improves the MAPE from 5.13% to
4.39%, and the time within 10% from 86% to 92%?

<details><summary>Answer</summary>

The trend is the same, and the self-calibrating digital twin is even more accurate than the best fixed r
(MAPE 3.6% vs. 4.5%).
The numbers differ because of the method: the paper computes the MAPE on 5-minute averages, after aligning the two
series in time, and tests 10 values of r; we use 1-minute samples, as the Self-Calibrator does, and 7 values of r.
The paper also finds that calibration alleviates the under-estimation bias (from 85% to 66% of the predictions); here,
from 93% to 69% of the minutes.
</details>

*Question:* is calibration always better than no calibration? Run the next cell.

In [ ]:
worse = with_calibration > no_calibration
print(f"Calibration is worse than no calibration in {worse.sum()} of {len(worse)} hours.")

<details><summary>Answer</summary>

No. It would be tempting to conclude that using calibration is always beneficial. However, in some hours,
the r picked on the last hour does not fit the next hour (e.g., when the workload changes).
The OpenDT paper observes the same: there exist significant simulation-periods when the no-calibration technique
performs better than the OpenDT calibration.
This suggests calibration is important but its technique requires further attention.
</details>

*Exercise 1:* by default, the Self-Calibrator computes the MAPE over the last 24 hours (`mape_window_minutes: 1440`),
not over the last hour. Run the next cell, which recalibrates every hour on the last 24 hours. Is a longer window
better?

In [ ]:
last_24h = errors.rolling("24h").mean().resample("1h").last()  # MAPE of each r over the last 24 hours
with_24h = self_calibrate(hourly, last_24h.idxmin(axis=1))
print(f"MAPE {with_24h.mean():.1f}%, worse than no calibration in {(with_24h > no_calibration).sum()} hours")

<details><summary>Answer</summary>

No. The MAPE increases from 3.6% to 4.3%, and calibration is worse than no calibration in 20 hours instead
of 5: a longer window reacts later to drift.
</details>

*Exercise 2 (optional):* count how often each r is the best. What does this suggest about the grid?

In [ ]:
best_r.value_counts().sort_index()

Then, simulate r = 10, 11, and 12 (three more simulations of the full week), and replay the self-calibrating
digital twin with r from 3 to 12.

In [ ]:
wider = simulate_many("r", [10.0, 11.0, 12.0], until=TRACE_END)
errors_wide = pd.concat([errors, pd.DataFrame({r: error(run) for r, run in wider.items()})], axis=1)
hourly_wide = errors_wide.resample("1h").mean()
with_wide = self_calibrate(hourly_wide, hourly_wide.idxmin(axis=1))
print(f"r from 3 to 12: MAPE {with_wide.mean():.1f}%")
hourly_wide.idxmin(axis=1).value_counts().sort_index()

<details><summary>Answer</summary>

r = 9, the edge of the grid, is the best r in 75 of 168 hours, which suggests that the grid is too narrow.
With r up to 12, the MAPE improves from 3.6% to 3.3%, but r = 12 is now the best in 69 hours, again at the edge.
In these hours, OpenDT under-estimates the power draw even with r = 12.
</details>

## Calibration in the Live Twin

*Optional.* The calibrator section of `config/experiments/experiment_2.yaml` configures the same grid search:

```yaml
calibrator:
  calibrated_property: "cpuPowerModel.calibrationFactor"  # r
  min_value: 3
  max_value: 9
  linspace_points: 10
  max_parallel_workers: 4
  mape_window_minutes: 60
```

To run it (requires Docker), use `make up config=config/experiments/experiment_2.yaml`, then follow the Self-Calibrator with
`make logs-calibrator`: after each grid search, it logs the MAPE of each candidate, the best value
(`Best: cpuPowerModel.calibrationFactor=...`), and publishes it to the simulator (`Published topology with ...`).

## Summary

In this notebook, we (1) look at the power model and its calibration parameter r, (2) calibrate r with a grid search,
as the Self-Calibrator does, and (3) replay the self-calibrating digital twin over one week and compare it with no
calibration.

*Next:* use case 3, CO2-aware workload migration, in [3.co2_migration.ipynb](3.co2_migration.ipynb).

*Learn more:* the [calibration process](https://atlarge-research.github.io/opendt/docs/concepts/calibration), the [calibrator service](https://atlarge-research.github.io/opendt/docs/services/calibrator),
and the [OpenDT paper](https://atlarge-research.com/pdfs/2026-icpe-opendt.pdf) (Section 3.4).